## 说话人聚类

In [3]:
# 版本要求 modelscope version 升级至最新版本 funasr 升级至最新版本
from modelscope.pipelines import pipeline
sd_pipeline = pipeline(
    task='speaker-diarization',
    model='damo/speech_campplus_speaker-diarization_common',
    model_revision='v1.0.0'
)
input_wav = 'https://modelscope.cn/api/v1/models/damo/speech_campplus_speaker-diarization_common/repo?Revision=master&FilePath=examples/2speakers_example.wav'
result = sd_pipeline(input_wav)
print(result)
# 如果有先验信息，输入实际的说话人数，会得到更准确的预测结果
result = sd_pipeline(input_wav, oracle_num=2)
print(result)

2025-03-12 10:35:10,775 - modelscope - INFO - Use user-specified model revision: v1.0.0
2025-03-12 10:35:11,100 - modelscope - INFO - initiate model from /home/zhangjiayuan/.cache/modelscope/hub/damo/speech_campplus_speaker-diarization_common
2025-03-12 10:35:11,102 - modelscope - INFO - initiate model from location /home/zhangjiayuan/.cache/modelscope/hub/damo/speech_campplus_speaker-diarization_common.
2025-03-12 10:35:11,105 - modelscope - INFO - initialize model from /home/zhangjiayuan/.cache/modelscope/hub/damo/speech_campplus_speaker-diarization_common


KeyboardInterrupt: 

# asr模型下载及识别

In [7]:
from modelscope.pipelines import pipeline
from modelscope.utils.constant import Tasks

inference_pipeline = pipeline(
    task=Tasks.auto_speech_recognition,
    model='iic/speech_paraformer-large_asr_nat-zh-cn-16k-common-vocab8404-pytorch', 
    model_revision="v2.0.4",
    )

rec_result = inference_pipeline(input='https://isv-data.oss-cn-hangzhou.aliyuncs.com/ics/MaaS/ASR/test_audio/asr_example_zh.wav', output_dir="./", disable_pbar=True)
print(rec_result)

2025-03-17 11:53:52,602 - modelscope - INFO - Use user-specified model revision: v2.0.4
2025-03-17 11:53:52,915 - modelscope - INFO - initiate model from /home/zhangjiayuan/.cache/modelscope/hub/iic/speech_paraformer-large_asr_nat-zh-cn-16k-common-vocab8404-pytorch
2025-03-17 11:53:52,918 - modelscope - INFO - initiate model from location /home/zhangjiayuan/.cache/modelscope/hub/iic/speech_paraformer-large_asr_nat-zh-cn-16k-common-vocab8404-pytorch.
2025-03-17 11:53:52,922 - modelscope - INFO - initialize model from /home/zhangjiayuan/.cache/modelscope/hub/iic/speech_paraformer-large_asr_nat-zh-cn-16k-common-vocab8404-pytorch


funasr version: 1.2.2.
Check update of funasr, and it would cost few times. You may disable it by set `disable_update=True` in AutoModel
New version is available: 1.2.6.
Please use the command "pip install -U funasr" to upgrade.


2025-03-17 11:53:59,959 - modelscope - WARNING - No preprocessor field found in cfg.
2025-03-17 11:53:59,963 - modelscope - WARNING - No val key and type key found in preprocessor domain of configuration.json file.
2025-03-17 11:53:59,964 - modelscope - WARNING - Cannot find available config to build preprocessor at mode inference, current config: {'model_dir': '/home/zhangjiayuan/.cache/modelscope/hub/iic/speech_paraformer-large_asr_nat-zh-cn-16k-common-vocab8404-pytorch'}. trying to build by task and model information.
2025-03-17 11:53:59,965 - modelscope - WARNING - No preprocessor key ('funasr', 'auto-speech-recognition') found in PREPROCESSOR_MAP, skip building preprocessor.


key: asr_example_zh, text: 欢迎大家来体验达摩院推出的语音识别模型, score: -1.6472185850143433
[{'key': 'asr_example_zh', 'text': '欢迎大家来体验达摩院推出的语音识别模型', 'score': -1.6472185850143433, 'wd_score': [0.9250594973564148, 0.9361027479171753, 0.8952967524528503, 0.8971680402755737, 0.8945184350013733, 0.9131760597229004, 0.9476251006126404, 0.9358212947845459, 0.9448878169059753, 0.9279370903968811, 0.9370487928390503, 0.8969293236732483, 0.8957635760307312, 0.9339223504066467, 0.9308863282203674, 0.9434126615524292, 0.9074532389640808, 0.9311461448669434, 0.9277999997138977]}]


In [9]:
rec_result = inference_pipeline(input='https://isv-data.oss-cn-hangzhou.aliyuncs.com/ics/MaaS/ASR/test_audio/asr_example_zh.wav', 
                                output_dir="./", 
                                disable_pbar=True)
print(rec_result)

key: asr_example_zh, text: 欢迎大家来体验达摩院推出的语音识别模型, score: -1.6465388536453247


In [3]:
import numpy as np
print(np.exp(rec_result[0]['score']))

0.1925848226787211


# 分析实时模型中的chunk_size作用

In [ ]:
from funasr import AutoModel

chunk_size = [0, 10, 5] #[0, 10, 5] 600ms, [0, 8, 4] 480ms
encoder_chunk_look_back = 4 #number of chunks to lookback for encoder self-attention
decoder_chunk_look_back = 1 #number of encoder chunks to lookback for decoder cross-attention

model = AutoModel(model="paraformer-zh-streaming", model_revision="v2.0.4")

import soundfile
import os

wav_file = os.path.join(model.model_path, "example/asr_example.wav")
speech, sample_rate = soundfile.read(wav_file)
chunk_stride = chunk_size[1] * 960 # 600ms

cache = {}
total_chunk_num = int(len((speech)-1)/chunk_stride+1)
for i in range(total_chunk_num):
    speech_chunk = speech[i*chunk_stride:(i+1)*chunk_stride]
    is_final = i == total_chunk_num - 1
    res = model.generate(input=speech_chunk, cache=cache, is_final=is_final, chunk_size=chunk_size, encoder_chunk_look_back=encoder_chunk_look_back, decoder_chunk_look_back=decoder_chunk_look_back)
    print(res)

funasr version: 1.2.2.
Check update of funasr, and it would cost few times. You may disable it by set `disable_update=True` in AutoModel


New version is available: 1.2.4.
Please use the command "pip install -U funasr" to upgrade.


2025-03-05 08:10:25,952 - modelscope - INFO - Use user-specified model revision: v2.0.4
rtf_avg: 584.827: 100%|██████████| 1/1 [06:00<00:00, 360.17s/it]                                                                                               

[{'key': 'rand_key_2yW4Acq9GFz6Y', 'text': ''}]


rtf_avg: 0.132: 100%|██████████| 1/1 [00:00<00:00, 11.88it/s]


...

[{'key': 'rand_key_6KopZ9jZICffu', 'text': '音识'}]


rtf_avg: 0.155: 100%|██████████| 1/1 [00:00<00:00, 10.24it/s]


[{'key': 'rand_key_4G7FgtJsThJv0', 'text': '别模型'}]


rtf_avg: 0.483: 100%|██████████| 1/1 [00:00<00:00, 10.74it/s]

[{'key': 'rand_key_7In9ZMJLsCfMZ', 'text': ''}]


# 下载QWen2-7B-Instruct 模型 用于小红书asr


In [ ]:
#模型下载
from modelscope import snapshot_download
# model_dir = snapshot_download('Qwen/Qwen2-7B-Instruct')


from modelscope import AutoModelForCausalLM, AutoTokenizer
device = "cuda" # the device to load the model onto

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2-7B-Instruct",
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2-7B-Instruct")

prompt = "Give me a short introduction to large language model."
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(device)

generated_ids = model.generate(
    model_inputs.input_ids,
    max_new_tokens=512
)
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

ImportError: Using `low_cpu_mem_usage=True` or a `device_map` requires Accelerate: `pip install 'accelerate>=0.26.0'`